# Exercicio Pratico: CRISP-DM e Random Forest na Analise de Risco de Credito
## Laboratorio de Ciencia de Dados (Google Colab / Jupyter)

---

### Instrucoes para o Estudante
Neste laboratorio pratico, voce atuara como Cientista de Dados em uma Fintech de credito e executara o ciclo completo da metodologia CRISP-DM (Cross-Industry Standard Process for Data Mining).

O banco de dados fornecido contem desafios reais de producao:
1. Valores ausentes (Missing Values) em renda e score de credito.
2. Outliers espurios (erros de digitacao, idades negativas ou irreais).
3. Variaveis irrelevantes / Ruido puro (numeros aleatorios, identificadores hash).
4. Strings despadronizadas em variaveis categoricas.
5. Desbalanceamento de classes (inadimplentes sao minoria, mas causam grande impacto financeiro).

Siga as 6 fases do CRISP-DM abaixo. Onde houver '# TODO:', leia as dicas comentadas e implemente sua solucao em Python.

--- 
## FASE 1: Business Understanding (Entendimento do Negocio)

### 1.1 Contexto e Desafio da Fintech
A empresa concede emprestimos pessoais online. O objetivo e aprovar o maior numero de bons pagadores e barrar proponentes com alto risco de inadimplencia.

### 1.2 Custo Assimetrico dos Erros
- Falso Negativo (Calote nao previsto): Prejuizo medio de R$ 10.000,00 por emprestimo nao honrado.
- Falso Positivo (Recusa indevida de bom pagador): Perda de oportunidade de juros de R$ 600,00.

Objetivo de Negocio: Maximizar o Recall da classe Inadimplente (1) para proteger o caixa da empresa, mantendo uma taxa saudavel de aprovacao.

### 1.3 Criterios de Sucesso do Projeto
- Recall de Inadimplencia >= 80%
- F1-Score >= 0.85
- ROC-AUC >= 0.90
- O modelo deve ser capaz de filtrar o ruido do dataset atraves da analise de Feature Importance.

In [ ]:
# =============================================================================
# ETAPA 1: Importacao de Bibliotecas
# =============================================================================
# TODO: Importe as bibliotecas necessarias para manipulacao, visualizacao e modelagem.
# DICA: Voce precisara de numpy (np), pandas (pd), matplotlib.pyplot (plt) e seaborn (sns).
# DICA: De sklearn.model_selection, importe train_test_split.
# DICA: De sklearn.ensemble, importe RandomForestClassifier.
# DICA: De sklearn.metrics, importe classification_report, confusion_matrix, accuracy_score, recall_score, f1_score, roc_auc_score.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuracao visual
sns.set_theme(style="whitegrid")
np.random.seed(42)

print("Bibliotecas importadas com sucesso!")

--- 
## FASE 2: Data Understanding (Entendimento dos Dados)

A celula abaixo simula a extracao do banco de dados com ruidos intencionais. Execute-a para gerar a base inicial `df_raw`.

In [ ]:
# =============================================================================
# ETAPA 2.1: Geracao da Base de Dados Sintetica Ruidosa (Execucao Automatica)
# =============================================================================
np.random.seed(42)
n_amostras = 3000

# 1. Variaveis Categoricas Despadronizadas
vinculos_brutos = ['CLT', ' CLT ', 'clt', 'Autonomo', 'AUTONOMO', 'PJ', 'Servidor_Publico', 'Estudante']
vinculos = np.random.choice(vinculos_brutos, size=n_amostras, p=[0.35, 0.10, 0.05, 0.20, 0.05, 0.12, 0.08, 0.05])
estados_civis = np.random.choice(['Solteiro', 'Casado', 'Divorciado', 'Viuvo'], size=n_amostras, p=[0.45, 0.40, 0.12, 0.03])
escolaridades = np.random.choice(['Medio', 'Superior', 'Pos_Graduacao', 'Fundamental'], size=n_amostras, p=[0.40, 0.42, 0.12, 0.06])

# 2. Atributos Numericos
idade = np.random.normal(loc=38.0, scale=11.0, size=n_amostras).astype(float)
renda_mensal = np.random.exponential(scale=4200.0, size=n_amostras) + 1400.0
valor_solicitado = np.random.uniform(2000.0, 35000.0, size=n_amostras)
numero_parcelas = np.random.choice([12, 24, 36, 48, 60], size=n_amostras, p=[0.2, 0.3, 0.25, 0.15, 0.10])
score_serasa = np.clip(np.random.normal(loc=620.0, scale=140.0, size=n_amostras), 150.0, 990.0)
num_consultas_recentes = np.random.poisson(lam=2.3, size=n_amostras)
atrasos_ultimos_12m = np.random.poisson(lam=0.8, size=n_amostras)

# 3. Variaveis de Ruido Puro / Irrelevantes
ruido_estocastico = np.random.normal(loc=0.0, scale=50.0, size=n_amostras)
numero_da_sorte_app = np.random.randint(1000, 9999, size=n_amostras)
ip_origem_hash = [f"IP-{hex(np.random.randint(100000, 999999))[2:].upper()}" for _ in range(n_amostras)]

# 4. Relacao Nao-Linear com Inadimplencia Real
parcela_mensal = valor_solicitado / numero_parcelas
comprometimento = parcela_mensal / (renda_mensal + 1e-5)
stress = (comprometimento * 4.5) + (atrasos_ultimos_12m * 0.85) + (num_consultas_recentes * 0.25) - ((score_serasa - 600.0) / 130.0) - ((renda_mensal - 4000.0) / 3500.0) + np.random.normal(0.0, 0.9, size=n_amostras)
prob_inad = 1.0 / (1.0 + np.exp(-(stress - 1.8)))
inadimplente = (np.random.rand(n_amostras) < prob_inad).astype(int)

# 5. Injecao de Glitches / Outliers Espurios e NaNs
idade[np.random.choice(n_amostras, size=4, replace=False)] = 999.0
idade[np.random.choice(n_amostras, size=4, replace=False)] = -15.0
renda_mensal[np.random.choice(n_amostras, size=6, replace=False)] = 999999.0
renda_mensal[np.random.choice(n_amostras, size=int(0.06 * n_amostras), replace=False)] = np.nan
score_serasa[np.random.choice(n_amostras, size=int(0.05 * n_amostras), replace=False)] = np.nan

df_raw = pd.DataFrame({
    'proponente_id': [f"PROP-{10000 + i}" for i in range(n_amostras)],
    'idade': np.round(idade, 1),
    'tipo_vinculo': vinculos,
    'estado_civil': estados_civis,
    'escolaridade': escolaridades,
    'renda_mensal': np.round(renda_mensal, 2),
    'valor_solicitado': np.round(valor_solicitado, 2),
    'numero_parcelas': numero_parcelas,
    'score_serasa': np.round(score_serasa, 1),
    'num_consultas_recentes': num_consultas_recentes,
    'atrasos_ultimos_12m': atrasos_ultimos_12m,
    'ruido_estocastico': np.round(ruido_estocastico, 2),
    'numero_da_sorte_app': numero_da_sorte_app,
    'ip_origem_hash': ip_origem_hash,
    'inadimplente': inadimplente
})

print(f"Dataset bruto gerado: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas.")

In [ ]:
# =============================================================================
# ETAPA 2.2: Exploracao Inicial dos Dados (EDA)
# =============================================================================
# TODO 1: Exiba as 5 primeiras linhas do dataframe usando .head()
# TODO 2: Verifique os tipos de dados e valores nao-nulos com .info()
# TODO 3: Calcule a contagem de valores ausentes (NaNs) por coluna com .isnull().sum()
# TODO 4: Verifique o desbalanceamento da coluna 'inadimplente' com .value_counts(normalize=True)

# DICA: Use display(df_raw.head()) para ver as primeiras linhas formatadas.
# DICA: Para conferir nulos: df_raw.isnull().sum()

# Seu codigo aqui:


In [ ]:
# =============================================================================
# ETAPA 2.3: Visualizacao de Outliers e Desbalanceamento
# =============================================================================
# TODO 1: Crie um grafico de contagem (sns.countplot) para a variavel alvo 'inadimplente'.
# TODO 2: Crie um boxplot (sns.boxplot) para a coluna 'idade' e observe os valores absurdos (ex: 999 e negativos).

# DICA: sns.countplot(data=df_raw, x='inadimplente')
# DICA: sns.boxplot(y=df_raw['idade'])
# Seu codigo aqui:


--- 
## FASE 3: Data Preparation (Preparacao dos Dados)

Nesta fase, voce deve aplicar tecnicas de limpeza e engenharia de dados:
1. Padronizar strings da coluna `tipo_vinculo` (remover espacos e colocar em maiusculas).
2. Tratar outliers espurios (idades fora da faixa de 18 a 100 anos devem virar `np.nan`; rendas acima de R$ 200.000,00 tambem).
3. Engenharia de Atributos (Feature Engineering):
   - Criar `valor_parcela = valor_solicitado / numero_parcelas`
   - Criar `comprometimento_renda = valor_parcela / (renda_mensal + 1e-5)`
4. Tratar valores ausentes (`NaN`): Preencher variaveis numericas com a mediana usando `.fillna()`.
5. Descartar identificadores sem poder preditivo: Remover `proponente_id` e `ip_origem_hash`.
6. Codificar variaveis categoricas usando `pd.get_dummies()`.
7. Dividir em Treino (75%) e Teste (25%) com estratificacao (`stratify=y`).

In [ ]:
# =============================================================================
# ETAPA 3.1: Limpeza, Tratamento de Outliers e Feature Engineering
# =============================================================================
df_clean = df_raw.copy()

# TODO 1: Padronize a coluna 'tipo_vinculo' usando .str.strip().str.upper()
# DICA: df_clean['tipo_vinculo'] = df_clean['tipo_vinculo'].astype(str).str.strip().str.upper()

# TODO 2: Converta idades fora de [18, 100] e rendas > 200000 em np.nan
# DICA: df_clean.loc[(df_clean['idade'] < 18) | (df_clean['idade'] > 100), 'idade'] = np.nan
# DICA: df_clean.loc[df_clean['renda_mensal'] > 200000.0, 'renda_mensal'] = np.nan

# TODO 3: Crie as novas features: 'valor_parcela' e 'comprometimento_renda'
# DICA: df_clean['valor_parcela'] = df_clean['valor_solicitado'] / df_clean['numero_parcelas']
# DICA: df_clean['comprometimento_renda'] = df_clean['valor_parcela'] / (df_clean['renda_mensal'] + 1e-5)

# TODO 4: Remova as colunas 'proponente_id' e 'ip_origem_hash' com .drop(columns=[...])

# Seu codigo aqui:


In [ ]:
# =============================================================================
# ETAPA 3.2: Imputacao de Nulos e One-Hot Encoding
# =============================================================================
# TODO 1: Preencha os valores ausentes (NaNs) das colunas numericas com a mediana (.fillna())
# DICA: cols_num = ['idade', 'renda_mensal', 'score_serasa', 'comprometimento_renda']
# DICA: for col in cols_num: df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# TODO 2: Aplique pd.get_dummies nas variaveis categoricas ('tipo_vinculo', 'estado_civil', 'escolaridade') com drop_first=True
# DICA: df_encoded = pd.get_dummies(df_clean, drop_first=True)

# Seu codigo aqui:


In [ ]:
# =============================================================================
# ETAPA 3.3: Separacao de X, y e Divisao Treino/Teste
# =============================================================================
# TODO 1: Separe os preditores (X) e o alvo (y = 'inadimplente')
# TODO 2: Divida em treino (75%) e teste (25%) com train_test_split(..., test_size=0.25, random_state=42, stratify=y)

# DICA: X = df_encoded.drop(columns=['inadimplente'])
# DICA: y = df_encoded['inadimplente']
# DICA: X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Seu codigo aqui:


--- 
## FASE 4: Modeling (Modelagem com Random Forest)

### Por que o Random Forest e ideal para dados ruidosos?
1. Ensemble e Bagging: Combina multiplas arvores de decisao treinadas em amostras aleatorias (Bootstrap). O ruido pontual de uma arvore e neutralizado pelo voto da maioria.
2. Amostragem de Features (max_features): A cada divisao de no, apenas uma fracao dos atributos e sorteada, impedindo que ruidos monopolizem as decisoes.
3. Pesos Balanceados (class_weight='balanced'): Ajusta as penalidades de acordo com a frequencia das classes para compensar o desbalanceamento.

In [ ]:
# =============================================================================
# ETAPA 4: Treinamento do Modelo Random Forest
# =============================================================================
# TODO 1: Instancie o RandomForestClassifier com:
#        - n_estimators=200
#        - max_depth=10
#        - class_weight='balanced' (Essencial para focar na classe minoritaria de calote)
#        - random_state=42
# TODO 2: Treine o modelo com .fit(X_train, y_train)

# DICA: modelo_rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)
# DICA: modelo_rf.fit(X_train, y_train)

# Seu codigo aqui:


--- 
## FASE 5: Evaluation (Avaliacao do Modelo e Impacto no Negocio)

Nesta fase, avalie se o modelo cumpriu os criterios do negocio:
1. Metricas de Classificacao: Gerar o `classification_report` e a matriz de confusao com `sns.heatmap`.
2. Feature Importance: Verificar se o Random Forest atribuiu importancia proxima de zero as variaveis de ruido puro (`ruido_estocastico`, `numero_da_sorte_app`).
3. ROI Financeiro: Calcular o prejuizo evitado versus o custo de falsos positivos.

In [ ]:
# =============================================================================
# ETAPA 5.1: Avaliacao de Metricas e Matriz de Confusao
# =============================================================================
# TODO 1: Obtenha as predicoes no conjunto de teste com modelo_rf.predict(X_test)
# TODO 2: Exiba o classification_report(y_test, y_pred)
# TODO 3: Plote a matriz de confusao usando sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
# TODO 4: Calcule a area sob a curva ROC com roc_auc_score(y_test, modelo_rf.predict_proba(X_test)[:, 1])

# DICA: y_pred = modelo_rf.predict(X_test)
# DICA: print(classification_report(y_test, y_pred))

# Seu codigo aqui:


In [ ]:
# =============================================================================
# ETAPA 5.2: Importancia das Variaveis (Feature Importance)
# =============================================================================
# TODO 1: Extraia as importancias das variaveis com modelo_rf.feature_importances_
# TODO 2: Monte um DataFrame com as colunas 'Atributo' (X_train.columns) e 'Importancia'
# TODO 3: Ordene os valores de forma decrescente e plote um grafico de barras (sns.barplot)
# TODO 4: Verifique a posicao das variaveis de ruido puro ('ruido_estocastico', 'numero_da_sorte_app')

# DICA:
# df_importancias = pd.DataFrame({'Atributo': X_train.columns, 'Importancia': modelo_rf.feature_importances_})
# df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)
# plt.figure(figsize=(10, 6))
# sns.barplot(data=df_importancias, x='Importancia', y='Atributo')
# plt.show()

# Seu codigo aqui:


In [ ]:
# =============================================================================
# ETAPA 5.3: Calculo do Retorno Financeiro (ROI do Modelo)
# =============================================================================
# TODO 1: Extraia os valores da matriz de confusao: vn, fp, fn, vp = confusion_matrix(y_test, y_pred).ravel()
# TODO 2: Calcule o custo operacional sem modelo: total_inadimplentes * R$ 10.000
# TODO 3: Calcule o custo com o modelo: (fn * 10000) + (fp * 600)
# TODO 4: Calcule a economia gerada e imprima os resultados

# DICA:
# custo_sem_modelo = (fn + vp) * 10000.0
# custo_com_modelo = (fn * 10000.0) + (fp * 600.0)
# economia = custo_sem_modelo - custo_com_modelo
# print(f"Economia Financeira Gerada: R$ {economia:,.2f}")

# Seu codigo aqui:


--- 
## FASE 6: Deployment (Implantacao da Funcao de Decisao)

Simule a implantacao criando uma funcao que recebe um dicionario com os dados de um novo proponente e retorna a decisao de credito instantanea.

In [ ]:
# =============================================================================
# ETAPA 6: Funcao de Predicao em Producao
# =============================================================================
# TODO: Crie a funcao analisar_proposta(payload_proponente, modelo, colunas_treinamento)
#       1. Converta o dicionario em DataFrame (pd.DataFrame([payload_proponente]))
#       2. Calcule 'valor_parcela' e 'comprometimento_renda'
#       3. Aplique pd.get_dummies e garanta que tenha exatamente as mesmas colunas de X_train com .reindex(columns=colunas_treinamento, fill_value=0)
#       4. Calcule a probabilidade de inadimplencia com modelo.predict_proba(...)[0, 1]
#       5. Retorne se a proposta foi APROVADA (probabilidade < 0.35) ou REPROVADA

# DICA:
# def analisar_proposta(payload, modelo, colunas_esperadas):
#     df_p = pd.DataFrame([payload])
#     df_p['valor_parcela'] = df_p['valor_solicitado'] / df_p['numero_parcelas']
#     df_p['comprometimento_renda'] = df_p['valor_parcela'] / (df_p['renda_mensal'] + 1e-5)
#     df_p = pd.get_dummies(df_p)
#     df_p = df_p.reindex(columns=colunas_esperadas, fill_value=0)
#     prob = modelo.predict_proba(df_p)[0, 1]
#     return {
#         'probabilidade_inadimplencia': f"{prob*100:.1f}%",
#         'decisao': 'APROVADO' if prob < 0.35 else 'REPROVADO'
#     }

# Teste com um proponente:
proposta_teste = {
    'idade': 35,
    'tipo_vinculo': 'CLT',
    'estado_civil': 'Casado',
    'escolaridade': 'Superior',
    'renda_mensal': 7500.0,
    'valor_solicitado': 9000.0,
    'numero_parcelas': 24,
    'score_serasa': 790.0,
    'num_consultas_recentes': 1,
    'atrasos_ultimos_12m': 0,
    'ruido_estocastico': 5.2,
    'numero_da_sorte_app': 3141
}

# Seu codigo aqui:
